# Lab 4: A Team of Agents
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/notebooks/04_multi_agent.ipynb)

Some jobs are easier for a **team** of focused agents than for one agent doing everything.
Our team will plan a personal conference day:

| Agent | Job | Uses an LLM? |
|---|---|---|
| **Planner** | Turns a request into a plan: which day, which topics | Yes |
| **Researcher** | Finds sessions for each topic (your Lab 3 agent!) | Yes, with tools |
| **Critic** | Checks the draft agenda for clashes and removes conflicts | Code checks facts, LLM judges |
| **Writer** | Turns the final agenda into a friendly message (your exercise) | Yes |

**A word of caution:** more agents means more calls, more cost and more places to go wrong.
Use a team only when the task naturally splits into roles.

### Setup
Run the cell below. The last line tells you which **brain** is answering:
`openai` or `gemini` (real AI in the cloud), `ollama` (AI on your own computer) or `mock`
(offline practice brain). All of them work for every exercise in this workshop.

In Colab, the first time a notebook reads your key, a pop-up asks for permission:
click **Grant access**. Every lab asks once.

In [ ]:
# SETUP: run this cell first. It is safe to run more than once.
import os, sys, subprocess
REPO_URL = "https://github.com/bmdayal/from-prompt-to-agent"

if os.path.isdir("../workshop"):        # running locally from notebooks/ or solutions/
    sys.path.insert(0, "..")
elif os.path.isdir("workshop"):         # running from the repo folder
    sys.path.insert(0, ".")
else:                                   # running in Google Colab: download the workshop files
    if not os.path.isdir("from-prompt-to-agent"):
        print("Downloading workshop files...")
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    sys.path.insert(0, "from-prompt-to-agent")

import json
from workshop import llm

PROVIDER = "auto"   # "auto", "openai", "gemini", "ollama" or "mock" (offline practice brain)
llm.configure(PROVIDER)

In [ ]:
from workshop.agent import run_agent
from workshop.tools import check_conflicts, get_session_details

# Real models on a free tier allow only a few calls per minute.
# Set this to False to make the Researcher use the search tool directly (fewer AI calls).
RESEARCHER_USES_LLM = True

## Agent 1: the Planner
It reads a free-form request and returns a structured plan. We **validate** its output
before trusting it (a habit you will see again in Lab 5).

In [ ]:
PLANNER_SYSTEM = """You are the PLANNER for a conference assistant.
Read the attendee's request and extract the day they attend and their interests, most important first.
Reply ONLY with JSON: {"day": "Thursday" or "Friday" or "Saturday", "topics": ["short topic words, e.g. agents, security, healthcare"]}"""

def planner(request):
    plan = llm.parse_json(llm.ask(request, system=PLANNER_SYSTEM, json_mode=True))
    if not isinstance(plan, dict) or not plan.get("topics"):
        print("  Planner reply was unusable, using a safe default plan.")
        plan = {"day": "Thursday", "topics": ["agents"]}
    plan["topics"] = [t.lower() for t in plan["topics"]][:4]
    return plan

request = "I'm attending on Thursday. I'm interested in AI agents, security and healthcare. Plan my day!"
plan = planner(request)
print("PLAN:", plan)

## Agent 2: the Researcher
For each topic, the Researcher runs the agent from Lab 3. Notice that we collect sessions from the
**tool results**, not from the model's prose: tool output is the source of truth.

In [ ]:
def researcher(topic, day):
    if not RESEARCHER_USES_LLM:
        from workshop.tools import search_sessions
        return search_sessions(topic, day)
    _, messages = run_agent(f"Search the program for {topic} sessions on {day} and list their ids.", verbose=False)
    found = []
    for m in messages:
        if m["role"] == "user" and m["content"].startswith("Observation:"):
            data = llm.parse_json(m["content"].split(":", 1)[1])
            items = data if isinstance(data, list) else [data] if isinstance(data, dict) else []
            found += [s for s in items if isinstance(s, dict) and "id" in s and "title" in s]
    return found

draft = []
for topic in plan["topics"]:
    for s in researcher(topic, plan["day"]):
        if s["id"] not in [d["id"] for d in draft]:
            draft.append({"id": s["id"], "title": s["title"], "start": s["start"],
                          "end": s["end"], "room": s["room"], "topic": topic})
draft.sort(key=lambda s: s["start"])

print(f"DRAFT AGENDA ({len(draft)} sessions):")
for s in draft:
    print(f'  {s["id"]}  {s["start"]}-{s["end"]}  [{s["topic"]}]  {s["title"]}')

**Predict:** will this draft work for a real person? Look at the times.

## Agent 3: the Critic
Checking times is a job for **code**, which is never wrong about arithmetic. Deciding **which**
session to drop is a judgment call, so that part goes to the **LLM**. Then we verify the LLM's
suggestion before applying it.

In [ ]:
CRITIC_SYSTEM = """You are the CRITIC. You review a draft conference agenda.
The CONFLICT REPORT was computed by code and is always correct.
For each conflict, remove the session whose topic is LOWER in the INTEREST ORDER.
Reply ONLY with JSON: {"approved": true or false, "issues": ["short descriptions"], "remove": ["session ids to remove"]}"""

def critic(draft, interests):
    report = check_conflicts([s["id"] for s in draft])        # facts, from code
    prompt = (f"INTEREST ORDER: {json.dumps(interests)}\n"
              f"DRAFT: {json.dumps(draft)}\n"
              f"CONFLICT REPORT: {json.dumps(report)}")
    verdict = llm.parse_json(llm.ask(prompt, system=CRITIC_SYSTEM, json_mode=True)) or {}
    return report, verdict

agenda = list(draft)
for round_no in range(1, 4):                                   # bounded, like max_steps
    report, verdict = critic(agenda, plan["topics"])
    print(f"Round {round_no}: {len(report['conflicts'])} conflict(s). Critic says: {verdict}")
    if report["ok"]:
        print("Approved: no conflicts left.")
        break
    ids_in_agenda = {s["id"] for s in agenda}
    to_remove = [i for i in verdict.get("remove", []) if i in ids_in_agenda]   # trust, but verify
    if not to_remove:                                                          # safety net
        to_remove = [report["conflicts"][0]["b"]]
        print("  Critic gave no usable suggestion, removing", to_remove, "by default.")
    agenda = [s for s in agenda if s["id"] not in to_remove]

print("\nFINAL AGENDA:")
for s in agenda:
    print(f'  {s["start"]}-{s["end"]}  {s["title"]}  ({s["room"]})')

## Your turn: Agent 4, the Writer
Write the instructions (system prompt) for a **Writer** agent that turns the final agenda into a
short, friendly message for the attendee. Keep the first sentence `You are the WRITER.` as is.

In [ ]:
WRITER_SYSTEM = """You are the WRITER.
TODO: describe the tone, length and format you want."""

def writer(agenda):
    return llm.ask(f"AGENDA: {json.dumps(agenda)}", system=WRITER_SYSTEM)

print(writer(agenda))

## The whole team in one function
This is an **orchestrator**: plain code that passes structured data from one agent to the next.

In [ ]:
def plan_my_day(request):
    plan = planner(request)
    print("Planner:", plan)
    draft = []
    for topic in plan["topics"]:
        for s in researcher(topic, plan["day"]):
            if s["id"] not in [d["id"] for d in draft]:
                draft.append({**{k: s[k] for k in ("id", "title", "start", "end", "room")}, "topic": topic})
    draft.sort(key=lambda s: s["start"])
    print("Researcher found:", [s["id"] for s in draft])
    for _ in range(3):
        report, verdict = critic(draft, plan["topics"])
        if report["ok"]:
            break
        remove = [i for i in verdict.get("remove", []) if i in {s["id"] for s in draft}] or [report["conflicts"][0]["b"]]
        draft = [s for s in draft if s["id"] not in remove]
    print("Critic approved:", [s["id"] for s in draft], "\n")
    return writer(draft)

print(plan_my_day("I'll be there Friday. I love anything about agents and evaluation, and a bit of security."))

**Vote:** which agent should we add next? A *Navigator* (walking time between rooms), a
*Networking* agent (who to meet), or a *Reminder* agent (calendar invites)?

## Key takeaways
- Give each agent **one focused job** and a short, clear prompt.
- Pass **structured data** (JSON) between agents, not long prose.
- Let **code check facts** and the **LLM handle judgment and language**.
- **Bound every loop**, and always verify an agent's output before acting on it.

**Next: Lab 5, Break It (and Fix It).** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/notebooks/05_break_it.ipynb)

Click the badge to open the next lab in a new Colab tab, then run its setup cell first. (Each lab is self-contained, so you can close this one.)